In [ ]:
import http.client
import json
import csv
import time
import pandas as pd
import os

API_KEY = "04ee69ca5ff23dd4b41709ae88ae8b4a"
HEADERS = {
    'x-apisports-key': API_KEY
}

# konversi b. indo ke b. ing
COUNTRY_DICTIONARY = {
    "Amerika": "USA", "Spanyol": "Spain", "Prancis": "France", "Inggris": "England",
    "Belanda": "Netherlands", "Belgia": "Belgium", "Jerman": "Germany", "Kroasia": "Croatia",
    "Maroko": "Morocco", "Swiss": "Switzerland", "Jepang": "Japan", "Korea Selatan": "South Korea",
    "Ekuador": "Ecuador", "Norwegia": "Norway", "Mesir": "Egypt", "Skotlandia": "Scotland",
    "Pantai Gading": "Ivory Coast", "Arab Saudi": "Saudi Arabia", "Afrika Selatan": "South Africa",
    "Selandia Baru": "New Zealand", "Bosnia dan Herzegovina": "Bosnia", "Kosta Rika": "Costa Rica",
    "Wales": "Wales"
}

def request_api(endpoint):
    conn = http.client.HTTPSConnection("v3.football.api-sports.io")
    formatted_endpoint = endpoint.replace(" ", "%20")
    conn.request("GET", formatted_endpoint, headers=HEADERS)
    res = conn.getresponse()
    data = res.read()

    if res.status != 200:
        print(f"HTTP Error {res.status}: {res.reason} on {endpoint}")
        return {}

    try:
        return json.loads(data.decode("utf-8"))
    except json.JSONDecodeError:
        print(f"Gagal mem-parsing JSON dari {endpoint}.")
        return {}

def get_national_team_id(country_name):
    search_name = COUNTRY_DICTIONARY.get(country_name, country_name)
    print(f"\nMencari ID untuk timnas: {country_name} (Search: {search_name})...")

    endpoint = f"/teams?name={search_name}"
    response = request_api(endpoint)

    if response and response.get("response"):
        for item in response["response"]:
            if item["team"]["national"] == True:
                return item["team"]["id"]
    return None

def fetch_and_save_squad_with_rating(team_id, country_name, fifa_ranking, filename):
    print(f"Mengambil data pemain & rating untuk ID {team_id}...")

    all_players = []
    page = 1
    total_pages = 1

    # Endpoint /players dipaginasi, kita harus looping halamannya
    while page <= total_pages:
        # Season 2022 dipilih agar relevan dengan World Cup
        endpoint = f"/players?team={team_id}&season=2022&page={page}"
        response = request_api(endpoint)

        if response and response.get("response"):
            if page == 1:
                total_pages = response.get("paging", {}).get("total", 1)

            for item in response["response"]:
                player = item["player"]
                # Ambil objek statistik utama (berisi rating, posisi, nomor punggung)
                stats = item["statistics"][0] if item.get("statistics") else {}
                games = stats.get("games", {})

                rating = games.get("rating")

                all_players.append({
                    "id": player.get("id"),
                    "name": player.get("name"),
                    "age": player.get("age"),
                    "number": games.get("number", "N/A"),
                    "position": games.get("position", "Unknown"),
                    "photo": player.get("photo"),
                    "player_rating": rating if rating else "N/A",
                    "fifa_country_ranking": fifa_ranking
                })
        else:
            print(f"⚠️ Gagal/Data kosong di page {page} untuk ID {team_id}")
            break

        page += 1

        # Jeda antar-halaman API (karena 1 tim bisa 2-3 halaman)
        if page <= total_pages:
            time.sleep(6.5)

    if all_players:
        header = ["id", "name", "age", "number", "position", "photo", "player_rating", "fifa_country_ranking"]

        with open(filename, mode='w', newline='', encoding='utf-8') as file:
            writer = csv.DictWriter(file, fieldnames=header)
            writer.writeheader()
            writer.writerows(all_players)
        print(f"✅ Berhasil menyimpan {len(all_players)} pemain ke {filename}")

# ==========================================
# Mulai Proses
# ==========================================
print("Membaca file pesertaWorldCup.csv...")
df_countries = pd.read_csv("/content/pesertaWorldCup.csv")
print(f"Total negara yang akan diproses: {len(df_countries)}\n")

for index, row in df_countries.iterrows():
    country = str(row['Nama_Negara'])
    if pd.isna(country) or country == "nan":
        continue

    fifa_ranking = row['Rangking_Fifa']
    filename = f"squad_timnas_{country.replace(' ', '_')}.csv"

    if os.path.exists(filename):
        print(f"⏩ Data {country} sudah ada ({filename}), melewati...")
        continue

    team_id = get_national_team_id(country)

    if team_id:
        # Beri jeda antar pencarian tim dan pengambilan pemain
        time.sleep(6.5)
        fetch_and_save_squad_with_rating(team_id, country, fifa_ranking, filename)
    else:
        print(f"❌ Tidak menemukan data Tim Nasional untuk {country}")

    # Jeda yang lebih lama antar negara
    time.sleep(12.0)

print("\n🎉 Semua proses selesai!")

Membaca file pesertaWorldCup.csv...
Total negara yang akan diproses: 57

⏩ Data USA sudah ada (squad_timnas_USA.csv), melewati...
⏩ Data Mexico sudah ada (squad_timnas_Mexico.csv), melewati...
⏩ Data Canada sudah ada (squad_timnas_Canada.csv), melewati...
⏩ Data Spain sudah ada (squad_timnas_Spain.csv), melewati...
⏩ Data Argentina sudah ada (squad_timnas_Argentina.csv), melewati...
⏩ Data France sudah ada (squad_timnas_France.csv), melewati...
⏩ Data England sudah ada (squad_timnas_England.csv), melewati...
⏩ Data Brazil sudah ada (squad_timnas_Brazil.csv), melewati...
⏩ Data Portugal sudah ada (squad_timnas_Portugal.csv), melewati...
⏩ Data Netherlands sudah ada (squad_timnas_Netherlands.csv), melewati...
⏩ Data Belgium sudah ada (squad_timnas_Belgium.csv), melewati...
⏩ Data Germany sudah ada (squad_timnas_Germany.csv), melewati...
⏩ Data Croatia sudah ada (squad_timnas_Croatia.csv), melewati...
⏩ Data Morocco sudah ada (squad_timnas_Morocco.csv), melewati...
⏩ Data Colombia sudah a

In [ ]:
@title Fungsi untuk membuat deskripsi pemain

def create_description(row):
    return (f"Pemain bernama {row['name']} berusia {row['age']} tahun. "
            f"Ia bermain di posisi {row['position']} untuk tim nasional {row['country']}. "
            f"Negara tersebut saat ini memiliki peringkat FIFA {row['fifa_country_ranking']}. "
            f"Rating performa pemain ini adalah {row['player_rating']}.")

# Menerapkan fungsi ke setiap baris
df['ml_text'] = df.apply(create_description, axis=1)

# Menyimpan dalam format .txt tanpa header untuk training
df['ml_text'].to_csv('Data_Pemain_ML_Description.txt', index=False, header=False)